# Step 16.2: source participant to target species mean
Select a GPU runtime, then run all cells. Each selected source participant is compared separately with the target species mean RDMs. No permutations.
Build a new ZIP with create_cross_species_package.py --specie D --target_specie H --radius 5 --target_radius 4. The package fixes both radii, masks, participant selections and class.
Each participant produces only an HDF5 and a log under OUTPUT_ROOT/EmoC/results/RSA_cross-specie/basic-block/class_all-mahalanobis/D-sub-XX/. Matrix dimensions count only nonzero mask voxels, excluding air. Closed snapshots use the same HDF5 path and are marked incomplete until finished. Use one writer per output/staging directory.


In [ ]:
%pip -q install nibabel h5py PyYAML
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PACKAGE = '/content/drive/MyDrive/rsa_colab/cross_species_EmoC_D5_vs_H4.zip'
OUTPUT_ROOT = '/content/drive/MyDrive/rsa_colab/results_step16_2'
STAGING = '/content/cross_species_16_2_v2'
DEVICE = 'auto'  # automatically uses CUDA; 'cuda' requires GPU
SOURCE_BATCH = 128
TARGET_BATCH = 4096
CHECKPOINT_MINUTES = 15  # copies entire closed partial file; 0 copies every source batch
# Reduce batches on out-of-memory. Existing checkpoints retain their original tile sizes.
# Use new output/staging roots to change tile sizes or analysis inputs.


In [ ]:
import sys, zipfile, json, importlib
from pathlib import Path
bootstrap = Path(STAGING) / 'bootstrap'
bootstrap.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE) as archive:
    for name in ('cross_species_rsa.py', 'run_colab_cross_species.py'):
        (bootstrap / name).write_bytes(archive.read(name))
    manifest = json.loads(archive.read('manifest.json'))
print({key: manifest[key] for key in ('dataset', 'model', 'rsa_class', 'specie', 'target_specie', 'radius', 'target_radius', 'kendall_variant')})
print({s: [p['participant'] for p in manifest['species'][s]['participants']] for s in ('D', 'H')})
sys.path.insert(0, str(bootstrap))
import cross_species_rsa, run_colab_cross_species
importlib.reload(cross_species_rsa)
importlib.reload(run_colab_cross_species)
print('Selected device:', cross_species_rsa.pick_device(DEVICE))
results = run_colab_cross_species.run_colab(PACKAGE, OUTPUT_ROOT, STAGING, device=DEVICE, dog_batch=SOURCE_BATCH, human_batch=TARGET_BATCH, checkpoint_minutes=CHECKPOINT_MINUTES)
print('Completed participant matrices:', *results, sep='\n')


In [ ]:
# Inspect one participant result lazily; no NIfTI is written.
from cross_species_rsa import CorrelationReader
with CorrelationReader(results[0]) as reader:
    source = reader.file.attrs['source_specie']
    target = reader.file.attrs['target_specie']
    print('Masked matrix:', reader.file['correlations'].shape)
    print('Mask counts:', len(reader.file[f'{source}/ijk']), len(reader.file[f'{target}/ijk']))
    source_ijk = reader.file[f'{source}/ijk'][0]
    target_image = reader.target_map(source_ijk)
    print('Source centre:', source_ijk, 'target shape:', target_image.shape)
    # Later thresholds and summaries can iterate reader.iter_blocks().
